In [2]:
import pandas as pd
import numpy as np
import logging
from pathlib import Path


pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

logging.basicConfig(
    level=logging.INFO,
    filename='students.log',
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

In [3]:
CSV_PATH = Path("clean_students.csv")

try:
    if not CSV_PATH.exists():
        raise FileNotFoundError(f"Missing input file: {CSV_PATH}")

    df = pd.read_csv(CSV_PATH)
    logger.info("Loaded %s with %d rows", CSV_PATH, len(df))
except (FileNotFoundError, pd.errors.EmptyDataError, pd.errors.ParserError):
    logger.exception("Failed to load the student data")
    raise

df.head()

,name,score,grade
0,Aarav,88.0,B
1,Sita,76.0,B
2,Bikash,76.0,B
3,Kiran,91.0,A
4,Unknown,67.0,C


In [4]:
try:
    required_columns = {"name", "score"}
    missing_columns = required_columns.difference(df.columns)
    if missing_columns:
        raise KeyError(f"Missing required columns: {sorted(missing_columns)}")

    logger.info("Validated required columns for downstream transformations")
except KeyError:
    logger.exception("Schema validation failed")
    raise

df.head()

,name,score,grade
0,Aarav,88.0,B
1,Sita,76.0,B
2,Bikash,76.0,B
3,Kiran,91.0,A
4,Unknown,67.0,C


In [5]:
# Quick sanity checks
try:
    print(df.info())
    print("\nMissing values:\n", df.isnull().sum())

    # Ensure score is numeric
    df["score"] = pd.to_numeric(df["score"], errors="coerce")

    invalid_scores = df["score"].isna().sum()
    if invalid_scores:
        logger.warning("Dropping %d row(s) with invalid score values", invalid_scores)
        df = df.dropna(subset=["score"]).copy()
except KeyError:
    logger.exception("Score column is missing during data validation")
    raise

<class 'pandas.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   name    14 non-null     str    
 1   score   14 non-null     float64
 2   grade   14 non-null     str    
dtypes: float64(1), str(2)
memory usage: 468.0 bytes
None

Missing values:
 name     0
score    0
grade    0
dtype: int64


In [6]:
def assign_grade(score):
    if pd.isna(score):
        return np.nan

    if score >= 90:
        return "A"
    elif score >= 75:
        return "B"
    elif score >= 60:
        return "C"
    elif score >= 50:
        return "D"
    else:
        return "F"

In [7]:
try:
    df["grade"] = df["score"].apply(assign_grade)
except KeyError:
    logger.exception("Failed to assign grades because score is missing")
    raise

df[["score", "grade"]].head()

,score,grade
0,88.0,B
1,76.0,B
2,76.0,B
3,91.0,A
4,67.0,C


In [8]:
try:
    df["passed"] = df["score"] >= 50
except KeyError:
    logger.exception("Failed to calculate pass/fail status")
    raise



In [9]:
df[["score", "passed"]].head()

,score,passed
0,88.0,True
1,76.0,True
2,76.0,True
3,91.0,True
4,67.0,True


In [10]:
def score_category(score):
    if pd.isna(score):
        return np.nan

    if score >= 80:
        return "High"
    elif score >= 50:
        return "Medium"
    else:
        return "Low"

try:
    df["score_category"] = df["score"].apply(score_category)
except KeyError:
    logger.exception("Failed to assign score categories")
    raise

df[["score", "score_category"]].head()

,score,score_category
0,88.0,High
1,76.0,Medium
2,76.0,Medium
3,91.0,High
4,67.0,Medium


In [11]:
try:
    df["rank"] = df["score"].rank(ascending=False, method="dense").astype(int)
except KeyError:
    logger.exception("Failed to compute ranks because score is missing")
    raise

df.sort_values("rank").head()

,name,score,grade,passed,score_category,rank
10,Rekha,95.0,A,True,High,1
3,Kiran,91.0,A,True,High,2
0,Aarav,88.0,B,True,High,3
9,Gopal,83.0,B,True,High,4
1,Sita,76.0,B,True,Medium,5


In [12]:
try:
    grade_stats = df.groupby("grade")["score"].agg(
        count="count",
        mean="mean",
        min="min",
        max="max"
    ).reset_index()
except KeyError:
    logger.exception("Failed to compute grade statistics")
    raise

grade_stats

,grade,count,mean,min,max
0,A,2,93.000000,91.0,95.0
1,B,5,79.800000,76.0,88.0
2,C,3,67.666667,64.0,72.0
3,D,1,59.000000,59.0,59.0
4,F,3,28.000000,0.0,45.0


In [13]:
try:
    df = df.sort_values("rank").reset_index(drop=True)
except KeyError:
    logger.exception("Failed to sort students by rank")
    raise

df.head(10)

,name,score,grade,passed,score_category,rank
0,Rekha,95.0,A,True,High,1
1,Kiran,91.0,A,True,High,2
2,Aarav,88.0,B,True,High,3
3,Gopal,83.0,B,True,High,4
4,Sita,76.0,B,True,Medium,5
5,Sunita,76.0,B,True,Medium,5
6,Bikash,76.0,B,True,Medium,5
7,Bibek,72.0,C,True,Medium,6
8,Unknown,67.0,C,True,Medium,7
9,Kabita,64.0,C,True,Medium,8


In [14]:
try:
    output_path = Path("enriched_students.csv")
    df.to_csv(output_path, index=False)
    logger.info("File saved: %s", output_path)
except OSError:
    logger.exception("Failed to save enriched_students.csv")
    raise

In [15]:
df.head(5)

,name,score,grade,passed,score_category,rank
0,Rekha,95.0,A,True,High,1
1,Kiran,91.0,A,True,High,2
2,Aarav,88.0,B,True,High,3
3,Gopal,83.0,B,True,High,4
4,Sita,76.0,B,True,Medium,5
